# Interpolation Pipeline Debug Notebook

按训练脚本同样流程逐步验证：SEG-Y 读取 -> 球面扩散补偿 -> 均匀缺失 -> 切块 -> UNet 前向/训练。

In [ ]:
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

from tools.segy_read import read_regular_shots
from tools.preprocessing import spherical_divergence_correction, mask_traces, normalize
from tools.patching import patchify_uniform
from model import build_model
from utils import build_loss, build_metrics, train_one_epoch, evaluate, visualize_random_sample

torch.set_grad_enabled(True)

In [ ]:
# 1) 配置（与 scripts/train_interpolation_unet.py 对齐）
segy_path = Path('/data/liuqi/code/MAE/5d-transformer/data/SEGC3-45/SEG_45Shot_shots1-9.sgy')
traces_per_shot = 201
dt = 0.002
power = 1.2
uniform_stride = 2
patch_trace, patch_time = 128, 256
overlap = 0.5
max_shots = 12  # notebook 调试时建议先小规模

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

In [ ]:
# 2) 读取并预处理
shots, _ = read_regular_shots(segy_path, traces_per_shot=traces_per_shot, return_headers=False)
shots = shots[:max_shots].astype(np.float32)
shots = spherical_divergence_correction(shots, dt=dt, power=power)
shots, _ = normalize(shots, mode='max_abs', per='global')
masked, missing_mask = mask_traces(shots, mode='uniform', ratio=0.5, uniform_stride=uniform_stride)

print('shots shape        =', shots.shape)
print('masked shape       =', masked.shape)
print('missing mask shape =', missing_mask.shape, 'missing ratio=', missing_mask.mean())

In [ ]:
# 3) 切块（trace,time）=(128,256)，两个维度 overlap=0.5
x_patches, info_x = patchify_uniform(masked, patch_size=(patch_trace, patch_time), overlap=overlap, output_ndim=4)
y_patches, info_y = patchify_uniform(shots, patch_size=(patch_trace, patch_time), overlap=overlap, output_ndim=4)

print('x_patches:', x_patches.shape, x_patches.dtype)
print('y_patches:', y_patches.shape, y_patches.dtype)
assert x_patches.shape == y_patches.shape
assert x_patches.ndim == 4 and x_patches.shape[1] == 1

In [ ]:
# 4) 构建 DataLoader
rng = np.random.default_rng(42)
idx = np.arange(x_patches.shape[0])
rng.shuffle(idx)
n_val = max(1, int(0.1 * len(idx)))
train_idx, val_idx = idx[:-n_val], idx[-n_val:]

train_ds = TensorDataset(torch.from_numpy(x_patches[train_idx]), torch.from_numpy(y_patches[train_idx]))
val_ds = TensorDataset(torch.from_numpy(x_patches[val_idx]), torch.from_numpy(y_patches[val_idx]))
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

xb, yb = next(iter(train_loader))
print('batch x/y:', xb.shape, yb.shape)

In [ ]:
# 5) UNet 前向 + 指标
model = build_model({
    'type': 'unet',
    'params': {'in_channels': 1, 'out_channels': 1, 'base_channels': 32, 'depth': 4}
}).to(device)
loss_fn = build_loss({'type': 'mse', 'params': {'reduction': 'mean'}}).to(device)
metrics = build_metrics([
    {'name': 'snr', 'params': {'reduction': 'per_sample'}},
    {'name': 'psnr', 'params': {'data_range': 2.0, 'reduction': 'per_sample'}},
    {'name': 'ssim', 'params': {'data_range': 2.0}},
])

with torch.no_grad():
    pred = model(xb.to(device))
print('pred shape:', pred.shape)
print('one-step loss:', loss_fn(pred, yb.to(device)).item())
print({k: v(pred, yb.to(device)) for k, v in metrics.items()})

In [ ]:
# 6) 小步训练 smoke test（2 epoch）
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=2, eta_min=1e-6)

for ep in range(2):
    tr = train_one_epoch(
        model=model,
        loader=train_loader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        device=device,
        epoch=ep,
        scheduler=scheduler,
        grad_clip=1.0,
        log_interval=100,
        logger=None,
    )
    vl, vm = evaluate(model, val_loader, loss_fn, metrics, device)
    print(f'epoch={ep} train={tr["train"]:.6f} val={vl["val"]:.6f} metrics={vm}')

In [ ]:
# 7) 随机样本可视化
out = Path('results/notebook_debug/random_vis.png')
visualize_random_sample(model, val_loader, save_path=out, device=device, title='Notebook debug', seed=None)
print('saved:', out)